# Image Classification Dataset Splitting

**Student ID:** 25509225  
**Purpose:** Split the Image Classification dataset into train/validation/test sets using student ID as random seed for reproducibility.

This notebook will:
1. Read the original dataset from `data/25509225/Image_Classification/dataset/`
2. Split it into train (70%), valid (15%), test (15%)
3. Save the split dataset to `data/25509225/Image_Classification/split_dataset/`
4. Generate a splitting report in `outputs/`

## Cell 1: Import Dependencies

In [18]:
import os
import sys
import shutil
import random
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

print("✓ All dependencies imported successfully")

✓ All dependencies imported successfully


## Cell 2: Configuration

In [19]:
# === Dataset Splitting Configuration ===
STUDENT_ID = "25509225"
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

# Paths
BASE_PATH = project_root / "data" / STUDENT_ID / "Image_Classification"
ORIGINAL_DATASET_PATH = BASE_PATH / "dataset"
SPLIT_DATASET_PATH = BASE_PATH / "split_dataset"
OUTPUTS_PATH = project_root / "outputs"

# Supported image extensions
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

print(f"Configuration:")
print(f"  Student ID: {STUDENT_ID}")
print(f"  Train Ratio: {TRAIN_RATIO:.0%}")
print(f"  Valid Ratio: {VALID_RATIO:.0%}")
print(f"  Test Ratio: {TEST_RATIO:.0%}")
print(f"  Original Dataset: {ORIGINAL_DATASET_PATH}")
print(f"  Output Directory: {SPLIT_DATASET_PATH}")

Configuration:
  Student ID: 25509225
  Train Ratio: 70%
  Valid Ratio: 15%
  Test Ratio: 15%
  Original Dataset: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/dataset
  Output Directory: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset


## Cell 3: Helper Functions

In [20]:
def verify_ratios(train_ratio, valid_ratio, test_ratio):
    """Verify that split ratios sum to 1.0."""
    total = train_ratio + valid_ratio + test_ratio
    if abs(total - 1.0) > 0.01:
        raise ValueError(f"Split ratios must sum to 1.0, got {total}")
    print("✓ Ratios verified: sum = 1.0")


def get_image_files(class_path):
    """
    Get all image files in a class directory.
    
    Args:
        class_path: Path to class directory
        
    Returns:
        List of image file paths
    """
    image_files = []
    for f in os.listdir(class_path):
        if Path(f).suffix.lower() in IMAGE_EXTENSIONS:
            image_files.append(f)
    return sorted(image_files)


def split_class_images(images, student_id, train_ratio, valid_ratio, test_ratio):
    """
    Split images into train/valid/test sets.
    
    Args:
        images: List of image filenames
        student_id: Student ID for random seed
        train_ratio: Training data ratio
        valid_ratio: Validation data ratio
        test_ratio: Test data ratio
        
    Returns:
        Tuple of (train_images, valid_images, test_images)
    """
    # Shuffle with fixed seed for reproducibility
    random.seed(int(student_id))
    shuffled = images.copy()
    random.shuffle(shuffled)
    
    n_total = len(shuffled)
    n_train = int(n_total * train_ratio)
    n_valid = int(n_total * valid_ratio)
    
    train_images = shuffled[:n_train]
    valid_images = shuffled[n_train:n_train + n_valid]
    test_images = shuffled[n_train + n_valid:]
    
    return train_images, valid_images, test_images


def create_split_directory_structure(split_dataset_path):
    """Create the output directory structure."""
    splits = ['train', 'valid', 'test']
    for split in splits:
        split_path = split_dataset_path / split
        split_path.mkdir(parents=True, exist_ok=True)
        print(f"✓ Created directory: {split_path}")


def copy_images_to_split(source_dir, dest_dir, images):
    """
    Copy images from source to destination directory.
    
    Args:
        source_dir: Source directory path
        dest_dir: Destination directory path
        images: List of image filenames to copy
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    for img_name in images:
        src_path = source_dir / img_name
        dst_path = dest_dir / img_name
        
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
        else:
            print(f"⚠ Warning: File not found: {src_path}")


print("✓ All helper functions defined")

✓ All helper functions defined


## Cell 4: Perform Dataset Splitting

In [21]:
print("="*80)
print("IMAGE CLASSIFICATION DATASET SPLITTING")
print("="*80)
print(f"\nStudent ID: {STUDENT_ID}")
print(f"Original Dataset: {ORIGINAL_DATASET_PATH}")
print(f"Output Directory: {SPLIT_DATASET_PATH}")
print(f"Split Ratios - Train: {TRAIN_RATIO:.0%}, Valid: {VALID_RATIO:.0%}, Test: {TEST_RATIO:.0%}")
print("\n" + "-"*80)

# Verify ratios
verify_ratios(TRAIN_RATIO, VALID_RATIO, TEST_RATIO)

# Check if original dataset exists
if not ORIGINAL_DATASET_PATH.exists():
    raise FileNotFoundError(f"Original dataset not found at: {ORIGINAL_DATASET_PATH}")

# Get all classes
classes = sorted([d.name for d in ORIGINAL_DATASET_PATH.iterdir() if d.is_dir()])
print(f"\nFound {len(classes)} classes:")
for i, cls in enumerate(classes, 1):
    print(f"  {i}. {cls}")

print("\n" + "-"*80)
print("Processing each class...")
print("-"*80 + "\n")

# Create output directory structure
create_split_directory_structure(SPLIT_DATASET_PATH)

# Process each class
total_stats = {
    'classes': {},
    'totals': {'train': 0, 'valid': 0, 'test': 0, 'total': 0}
}

for cls in classes:
    cls_path = ORIGINAL_DATASET_PATH / cls
    
    # Get all images for this class
    images = get_image_files(cls_path)
    
    if not images:
        print(f"⚠ Warning: No images found for class '{cls}', skipping...")
        continue
    
    # Split images
    train_imgs, valid_imgs, test_imgs = split_class_images(
        images, STUDENT_ID, TRAIN_RATIO, VALID_RATIO, TEST_RATIO
    )
    
    # Copy images to respective directories
    for split_name, split_imgs in [('train', train_imgs), 
                                   ('valid', valid_imgs), 
                                   ('test', test_imgs)]:
        dest_dir = SPLIT_DATASET_PATH / split_name / cls
        copy_images_to_split(cls_path, dest_dir, split_imgs)
        
        # Update statistics
        total_stats['totals'][split_name] += len(split_imgs)
    
    total_stats['totals']['total'] += len(images)
    total_stats['classes'][cls] = {
        'total': len(images),
        'train': len(train_imgs),
        'valid': len(valid_imgs),
        'test': len(test_imgs)
    }
    
    print(f"✓ {cls:30s} | Total: {len(images):4d} | "
          f"Train: {len(train_imgs):4d} | Valid: {len(valid_imgs):4d} | Test: {len(test_imgs):4d}")

stats = total_stats

print("\n" + "="*80)
print("SPLITTING SUMMARY")
print("="*80)
print(f"\nTotal Images: {stats['totals']['total']}")
print(f"  Training:   {stats['totals']['train']} ({stats['totals']['train']/stats['totals']['total']:.1%})")
print(f"  Validation: {stats['totals']['valid']} ({stats['totals']['valid']/stats['totals']['total']:.1%})")
print(f"  Testing:    {stats['totals']['test']} ({stats['totals']['test']/stats['totals']['total']:.1%})")
print(f"\nOutput saved to: {SPLIT_DATASET_PATH}")
print("="*80)

IMAGE CLASSIFICATION DATASET SPLITTING

Student ID: 25509225
Original Dataset: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/dataset
Output Directory: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset
Split Ratios - Train: 70%, Valid: 15%, Test: 15%

--------------------------------------------------------------------------------
✓ Ratios verified: sum = 1.0

Found 10 classes:
  1. CRESTED KINGFISHER
  2. CROW
  3. EASTERN MEADOWLARK
  4. FAIRY BLUEBIRD
  5. HARLEQUIN QUAIL
  6. LAUGHING GULL
  7. PALILA
  8. PARADISE TANAGER
  9. RAINBOW LORIKEET
  10. TOWNSENDS WARBLER

--------------------------------------------------------------------------------
Processing each class...
--------------------------------------------------------------------------------

✓ Created directory: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset/train
✓ Created directory: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Clas

## Cell 5: Generate Split Report

In [22]:
report = {
    'timestamp': datetime.now().isoformat(),
    'student_id': STUDENT_ID,
    'split_ratios': {
        'train': TRAIN_RATIO,
        'valid': VALID_RATIO,
        'test': TEST_RATIO
    },
    'statistics': stats,
    'output_path': str(SPLIT_DATASET_PATH)
}

# Save report
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)
report_path = OUTPUTS_PATH / f"classification_split_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✓ Split report saved to: {report_path}")
print(f"\n{'='*80}")
print("SPLIT REPORT DETAILS")
print(f"{'='*80}")
print(f"\nStudent ID: {report['student_id']}")
print(f"Timestamp: {report['timestamp']}")
print(f"Output Path: {report['output_path']}")

print(f"\n{'-'*80}")
print("Split Ratios:")
print(f"{'-'*80}")
for split, ratio in report['split_ratios'].items():
    print(f"  {split.capitalize():12s}: {ratio:6.1%}")

print(f"\n{'-'*80}")
print("Overall Statistics:")
print(f"{'-'*80}")
totals = report['statistics']['totals']
print(f"  Total Images:  {totals['total']:6d}")
print(f"  Training:      {totals['train']:6d} ({totals['train']/totals['total']:6.1%})")
print(f"  Validation:    {totals['valid']:6d} ({totals['valid']/totals['total']:6.1%})")
print(f"  Testing:       {totals['test']:6d} ({totals['test']/totals['total']:6.1%})")

print(f"\n{'-'*80}")
print("Per-Class Breakdown:")
print(f"{'-'*80}")
print(f"{'Class Name':<30s} {'Total':>6s} {'Train':>6s} {'Valid':>6s} {'Test':>6s}")
print(f"{'-'*80}")
for cls, stats_cls in sorted(report['statistics']['classes'].items()):
    print(f"{cls:<30s} {stats_cls['total']:>6d} {stats_cls['train']:>6d} {stats_cls['valid']:>6d} {stats_cls['test']:>6d}")

print(f"\n{'='*80}")

✓ Split report saved to: /home/sagemaker-user/ML_CNN_A2/outputs/classification_split_report_20260514_035801.json

SPLIT REPORT DETAILS

Student ID: 25509225
Timestamp: 2026-05-14T03:58:01.434375
Output Path: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset

--------------------------------------------------------------------------------
Split Ratios:
--------------------------------------------------------------------------------
  Train       :  70.0%
  Valid       :  15.0%
  Test        :  15.0%

--------------------------------------------------------------------------------
Overall Statistics:
--------------------------------------------------------------------------------
  Total Images:    1589
  Training:        1109 ( 69.8%)
  Validation:       231 ( 14.5%)
  Testing:          249 ( 15.7%)

--------------------------------------------------------------------------------
Per-Class Breakdown:
---------------------------------------------------------

## Cell 6: Verify Split Integrity

In [23]:
print("="*80)
print("VERIFYING SPLIT INTEGRITY")
print("="*80)

errors = []

# Check if PIL is available
try:
    from PIL import Image
    pil_available = True
    print("\n✓ PIL/Pillow is available for image verification")
except ImportError:
    pil_available = False
    print("\n⚠ Warning: PIL/Pillow not installed. Skipping image content verification.")
    print("  Install with: pip install Pillow\n")

# Check each split directory
for split in ['train', 'valid', 'test']:
    split_path = SPLIT_DATASET_PATH / split
    
    if not split_path.exists():
        errors.append(f"Missing split directory: {split_path}")
        continue
    
    # Check each class
    classes_in_split = [d.name for d in split_path.iterdir() if d.is_dir()]
    
    for cls in classes_in_split:
        cls_path = split_path / cls
        images = get_image_files(cls_path)
        
        if not images:
            errors.append(f"No images found in {cls_path}")
        
        # Verify all images are valid (only if PIL is available)
        if pil_available:
            for img in images[:5]:  # Check first 5 images
                img_path = cls_path / img
                try:
                    with Image.open(img_path) as pil_img:
                        pil_img.verify()
                except Exception as e:
                    errors.append(f"Corrupted image: {img_path} - {str(e)}")

if errors:
    print(f"\n❌ Verification FAILED with {len(errors)} error(s):")
    for err in errors[:10]:  # Show first 10 errors
        print(f"  - {err}")
else:
    print("\n✅ Verification PASSED - All splits are valid!")

VERIFYING SPLIT INTEGRITY

✓ PIL/Pillow is available for image verification

✅ Verification PASSED - All splits are valid!


## Cell 7: Final Summary

In [24]:
if not errors:
    print("\n" + "="*80)
    print("✅ DATASET SPLITTING COMPLETED SUCCESSFULLY!")
    print("="*80)
    print(f"\nYou can now use the split dataset for training:")
    print(f"  Training:   {SPLIT_DATASET_PATH}/train/")
    print(f"  Validation: {SPLIT_DATASET_PATH}/valid/")
    print(f"  Testing:    {SPLIT_DATASET_PATH}/test/")
    print("\nThe dataset is compatible with torchvision.datasets.ImageFolder")
    print("="*80)
else:
    print("\n⚠ WARNING: Split verification failed. Please check the errors above.")


✅ DATASET SPLITTING COMPLETED SUCCESSFULLY!

You can now use the split dataset for training:
  Training:   /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset/train/
  Validation: /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset/valid/
  Testing:    /home/sagemaker-user/ML_CNN_A2/data/25509225/Image_Classification/split_dataset/test/

The dataset is compatible with torchvision.datasets.ImageFolder
